# Setup

We require `pandas` library for easily handling the BZK `data.json` and `rdflib` to build a database from the gnd geographic authority file. 

In [2]:
%pip install pandas rdflib rapidfuzz

Note: you may need to restart the kernel to use updated packages.


If not present yet, we also need to download the gnd authority file. The file contains names and variant names for many different places in German, which we can use to populate a lookup table for names.

In [3]:
from pathlib import Path
import subprocess

GND_AUTHORITY_URL = "https://data.dnb.de/opendata/authorities-gnd-geografikum_lds.ttl.gz"
GND_AUTHORITY_FILE = Path("authorities-gnd-geografikum_lds.ttl")

if not GND_AUTHORITY_FILE.exists():
    subprocess.run(["wget", GND_AUTHORITY_URL])
    subprocess.run(["gzip", "-d", Path(GND_AUTHORITY_URL).name])
    assert GND_AUTHORITY_FILE.exists()
    print("GND authority file downloaded!")
else: print("GND Authority file already exists!")


GND Authority file already exists!


Using `rdflib` we can parse the authority file. Parsing the authority file may take some time.

In [4]:
from rdflib import Graph
gnd_graph = Graph()
print("Parsing graph...")
gnd_graph.parse(GND_AUTHORITY_FILE)
print("Graph parsed!")

Parsing graph...
Graph parsed!


In [5]:
# Temporary query box; 
query_palestine = """
SELECT ?id ?o
WHERE {
    ?id ?nameType "Palestine"
        FILTER (?nameType IN (
            gndo:preferredNameForThePlaceOrGeographicName, 
            gndo:variantNameForThePlaceOrGeographicName)).
    ?id owl:sameAs ?o
}
"""

# Obtains all gnd triples concerning some place

# Saxony: https://d-nb.info/gnd/4051176-5
query1 = """
SELECT ?p ?o 
WHERE {
    <https://d-nb.info/gnd/4051176-5> ?p ?o.
}
"""

# Berlin: http://www.wikidata.org/entity/Q64
query = """
SELECT ?s
WHERE {
    ?s owl:sameAs <http://www.wikidata.org/entity/Q64>.
}
"""

qres = gnd_graph.query(query)

for row in qres:
    print(" ".join(map(str,row)))

https://d-nb.info/gnd/4005728-8


In [6]:
query = """
SELECT ?id ?geonameId
WHERE {
    ?id owl:sameAs ?geonameId FILTER (STRSTARTS(STR(?geonameId), "https://sws.geonames.org/")).
    ?id owl:sameAs ?otherId FILTER (?geonameId != ?otherId && STRSTARTS(STR(?otherId), "https://sws.geonames.org/")).
}
"""

qres = gnd_graph.query(query)

print("GND entries with multiple GeoName IDs:")
for row in qres:
    print(" ".join(map(str,row)))

GND entries with multiple GeoName IDs:
https://d-nb.info/gnd/7748848-9 https://sws.geonames.org/867811
https://d-nb.info/gnd/7748848-9 https://sws.geonames.org/494193


In [7]:
query = """
SELECT ?wikidataId (COUNT(?geonameId) AS ?conflicts)
WHERE {
    ?id owl:sameAs ?wikidataId FILTER (STRSTARTS(STR(?wikidataId), "http://www.wikidata.org/")).
    ?id owl:sameAs ?geonameId FILTER (STRSTARTS(STR(?geonameId), "https://sws.geonames.org/")).
}
GROUP BY ?wikidataId
HAVING (COUNT(?geonameId) > 1)
"""

qres = gnd_graph.query(query)


for row in qres:
    print(" ".join(map(str,row)))
#print("No multiple Wikidata IDs found for any GND entry.")

http://www.wikidata.org/entity/Q935744 2
http://www.wikidata.org/entity/Q41252 2
http://www.wikidata.org/entity/Q280857 2
http://www.wikidata.org/entity/Q112162802 2
http://www.wikidata.org/entity/Q1549411 2
http://www.wikidata.org/entity/Q1422630 2
http://www.wikidata.org/entity/Q1453319 2
http://www.wikidata.org/entity/Q48922596 2
http://www.wikidata.org/entity/Q696853 2
http://www.wikidata.org/entity/Q387396 2
http://www.wikidata.org/entity/Q326142 2
http://www.wikidata.org/entity/Q92212 2
http://www.wikidata.org/entity/Q146351 2
http://www.wikidata.org/entity/Q885372 2
http://www.wikidata.org/entity/Q156974 2
http://www.wikidata.org/entity/Q47796842 2
http://www.wikidata.org/entity/Q829198 2
http://www.wikidata.org/entity/Q630083 2
http://www.wikidata.org/entity/Q21162566 2
http://www.wikidata.org/entity/Q1483267 2
http://www.wikidata.org/entity/Q53127777 2
http://www.wikidata.org/entity/Q1475579 2
http://www.wikidata.org/entity/Q673016 2
http://www.wikidata.org/entity/Q297299 2
ht

To capture potential locations, we collect into a csv file all names (preferred or variant) of relevant places in the authority file. Many address fiels include city and state name, but many addresses only list the city when the city is likely to be known (german or very popular cities). Therefore we also collect cities that may be more relevant: large capital cities, german speaking cities and some other special cases. The gnd authority file does not distinguish which places are cities and also does not include information about which states belong in which country. Therefore wikidata is used to acquire the city list, country list^1 the state to country correspondence. The wikidata id can then be cross referenced with the gnd id using the `owl:sameAs` property, letting us obtain the preferred and variant names accordingly.

^1 Country or state lists could also be obtained from the gnd authority file

In [8]:
import re
import pandas as pd
from rdflib.namespace import Namespace

PREFERRED_NAME_URI = "https://d-nb.info/standards/elementset/gnd#preferredNameForThePlaceOrGeographicName"
VARIANT_NAME_URI = "https://d-nb.info/standards/elementset/gnd#variantNameForThePlaceOrGeographicName"
WIKIDATA_ENGLISH_LABEL = "wikidata-english-label"
WIKIDATA_GERMAN_LABEL = "wikidata-german-label"
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

WIKIDATA_CITIES_QUERY = """
SELECT DISTINCT ?geonameId ?gndId ?city ?cityLabel_en ?entityLabel_de ?stateGeoname ?countryGeoname WHERE {
  {
    {
      ?city wdt:P31 wd:Q108178728.
      ?city p:P1082 / psv:P1082 / wikibase:quantityAmount ?pop FILTER (?pop > 1000000)
    }
    UNION
    {
      ?city wdt:P31 / wdt:P279* wd:Q515.
      { {?city wdt:P37 wd:Q188} UNION {?city wdt:P31 wd:Q42744322} }.
      ?city p:P1082 / psv:P1082 / wikibase:quantityAmount ?pop FILTER (?pop > 25000)
    }
  }.
  SERVICE wikibase:label {bd:serviceParam wikibase:language "en,de".}
  FILTER NOT EXISTS {?city wdt:P576 ?endOfCity FILTER (YEAR(?endOfCity) < 1800)}.
  ?city wdt:P1566 ?geonameId.
  OPTIONAL {?city wdt:P227 ?gndId}.
  ?city wdt:P17 ?country FILTER NOT EXISTS {?country wdt:P576 ?endOfCountry FILTER (YEAR(?endOfCountry) < 1800)}.
  ?country wdt:P1566 ?countryGeoname.
  OPTIONAL{
    ?city wdt:P131+ ?state.
    ?state wdt:P31 / wdt:P279* wd:Q107390.
    ?state wdt:P17 ?country.
    ?state wdt:P1566 ?stateGeoname.
    FILTER NOT EXISTS {?state wdt:P576 ?endOfState FILTER (YEAR(?endOfState) < 1800)}
  }.
}
"""

WIKIDATA_STATES_QUERY = """
SELECT DISTINCT ?geonameId ?gndId ?state ?stateLabel_en ?stateLabel_de ?countryGeoname WHERE {
   ?state wdt:P31 / wdt:P279* wd:Q107390.
   SERVICE wikibase:label {bd:serviceParam wikibase:language "en,de".}
   ?state wdt:P1566 ?geonameId.
   OPTIONAL {?state wdt:P227 ?gndId}.
   ?state wdt:P17 ?country FILTER NOT EXISTS {?country wdt:P576 ?endOfCountry FILTER (YEAR(?endOfCountry) < 1800)}.
   ?country  wdt:P1566 ?countryGeoname.
   FILTER NOT EXISTS {?state wdt:P576 ?endOfState FILTER (YEAR(?endOfState) < 1800)}.
}
"""

WIKIDATA_COUNTRIES_QUERY = """
SELECT DISTINCT ?geonameId ?gndId ?country ?countryLabel_en ?countryLabel_de WHERE {
  ?country wdt:P31 wd:Q6256 FILTER NOT EXISTS {?country wdt:P576 ?endOfCountry FILTER (YEAR(?endOfCountry) < 1800)}.
  SERVICE wikibase:label {bd:serviceParam wikibase:language "en,de".}
   ?country  wdt:P1566 ?geonameId.
  OPTIONAL {?country wdt:P227 ?gndId}
}
"""

GND_GET_ALL_NAMES_QUERY = """
SELECT ?nameType ?name WHERE {
        <%(gndUri)s> ?nameType ?name
        FILTER (?nameType IN (
            gndo:preferredNameForThePlaceOrGeographicName, 
            gndo:variantNameForThePlaceOrGeographicName)).
}
"""

GND_GET_URI_QUERY = "SELECT ?gndUri WHERE {?gndUri owl:sameAs <%(otherId)s>.}"

CLEANUP_REGEX = re.compile(r"([,.]|\(.*\))")

def name_to_search_key(name): return CLEANUP_REGEX.sub("", name.lower()).strip()

def geoname_id_to_uri(geoname_id): return f"https://sws.geonames.org/{geoname_id}" if geoname_id is not None else None

def fetch_gnd_uri(gnd_id, wikidata_uri, geoname_id):
    if gnd_id != None:
        return f"https://d-nb.info/gnd/{gnd_id}"
    qres = gnd_graph.query(GND_GET_URI_QUERY % {"otherId" : wikidata_uri})
    qres = list(qres)
    if not qres:
        qres = gnd_graph.query(GND_GET_URI_QUERY % {"otherId" : geoname_id_to_uri(geoname_id)})
        qres = list(qres)
    if not qres:
        return None
    else:
        return str(qres[0][0])

def create_place_lookup_table():
    col_names = ["geonameId", "gndUri", "wikidataUri", "countryGeoname",
                        "stateGeoname", "placeType", "nameType", "name", "searchKey"]
    rows = []
    def populate_with_names(geoname_id, gnd_id, wikidata_uri, english_label, german_label, country_geoname, 
                          state_geoname, place_type):
        geoname_id = geoname_id_to_uri(geoname_id)
        gnd_id = str(gnd_id) if gnd_id else None
        wikidata_uri = str(wikidata_uri)
        country_geoname = geoname_id_to_uri(country_geoname)
        state_geoname = geoname_id_to_uri(state_geoname)
        gnd_uri = fetch_gnd_uri(gnd_id, wikidata_uri, geoname_id)
        if gnd_uri is None:
            names_qres = []
        else:
            names_qres = gnd_graph.query(GND_GET_ALL_NAMES_QUERY % {"gndUri" : gnd_uri})
        for name_type, name in names_qres:
            name_type = str(name_type)
            name = str(name)
            search_key = name_to_search_key(name)
            rows.append([geoname_id, gnd_id, wikidata_uri, country_geoname, 
                          state_geoname, place_type, name_type, name, search_key])
        else:
            for name_type, label in [(WIKIDATA_ENGLISH_LABEL, english_label), (WIKIDATA_GERMAN_LABEL, german_label)]:
                if not label: continue
                label = str(label)
                search_key = name_to_search_key(label)
                rows.append([geoname_id, gnd_id, wikidata_uri, country_geoname, 
                          state_geoname, place_type, name_type, label, search_key])
    
    wikidata = Graph("SPARQLStore")
    wikidata.bind("wd", Namespace("http://www.wikidata.org/entity/"))
    wikidata.bind("wdt", Namespace("http://www.wikidata.org/prop/direct/"))
    wikidata.bind("p", Namespace("http://www.wikidata.org/prop/"))
    wikidata.bind("psv", Namespace("http://www.wikidata.org/prop/statement/value/"))
    wikidata.bind("wikibase", Namespace("http://wikiba.se/ontology#"))
    wikidata.open(WIKIDATA_ENDPOINT)

    print(f"Populating country names...")
    country_qres = wikidata.query(WIKIDATA_COUNTRIES_QUERY)
    for geoname_id, gnd_id, wikidata_uri, english_label, german_label in country_qres:
        populate_with_names(geoname_id, gnd_id, wikidata_uri, english_label, german_label, 
                            geoname_id, None, "country")

    print(f"Populating state names...")
    states_qres = wikidata.query(WIKIDATA_STATES_QUERY)
    for geoname_id, gnd_id, wikidata_uri, english_label, german_label, country_geoname in states_qres:
        populate_with_names(geoname_id, gnd_id, wikidata_uri, english_label, german_label, 
                            country_geoname, geoname_id, "state")

    print(f"Populating city names...")
    cities_qres = wikidata.query(WIKIDATA_CITIES_QUERY)
    for geoname_id, gnd_id, wikidata_uri, english_label, german_label, state_geoname, country_geoname in cities_qres:
        populate_with_names(geoname_id, gnd_id, wikidata_uri, english_label, german_label, 
                            country_geoname, state_geoname, "city")
    
    wikidata.close()
    df = pd.DataFrame(data=rows, columns=col_names)
    

PLACE_NAMES_CSV = Path("place-name-lookup.csv")
if PLACE_NAMES_CSV.exists():
    place_names_lookup = pd.read_csv(PLACE_NAMES_CSV)
    print("Place name lookup table loaded!")
else:
    place_names_lookup = create_place_lookup_table()
    place_names_lookup.to_csv(PLACE_NAMES_CSV)
    print("Place name lookup table created!")

print(f"Number of lookup table rows: {len(place_names_lookup)}")

Place name lookup table loaded!
Number of lookup table rows: 7552


Obtaining geonames from wikidata can result in many conflicts as seen bellow. Use gnd for resolving?

In [9]:
FIELD = "geonameId"
conflicts = place_names_lookup[["gndUri", "wikidataUri", "geonameId", "placeType", "name"]]
conflicts = conflicts.merge(conflicts, how="inner", on=["placeType", "name"])
conflicts = conflicts[conflicts[FIELD + "_x"] != conflicts[FIELD + "_y"]].drop_duplicates()
print(f"{len(conflicts)} conflicts found.")
conflicts.merge(conflicts[["placeType", "name"]].sample(n=5), how="right")

1218 conflicts found.


,gndUri_x,wikidataUri_x,geonameId_x,placeType,name,gndUri_y,wikidataUri_y,geonameId_y
0,4067037-5,http://www.wikidata.org/entity/Q2999,https://sws.geonames.org/6556942,city,Arctaunum,4067037-5,http://www.wikidata.org/entity/Q2999,https://sws.geonames.org/3220897
1,4067037-5,http://www.wikidata.org/entity/Q2999,https://sws.geonames.org/3220897,city,Arctaunum,4067037-5,http://www.wikidata.org/entity/Q2999,https://sws.geonames.org/6556942
2,4045637-7,http://www.wikidata.org/entity/Q3046,https://sws.geonames.org/3220724,city,Pforzheim,4045637-7,http://www.wikidata.org/entity/Q3046,https://sws.geonames.org/6547441
3,4045637-7,http://www.wikidata.org/entity/Q3046,https://sws.geonames.org/6547441,city,Pforzheim,4045637-7,http://www.wikidata.org/entity/Q3046,https://sws.geonames.org/3220724
4,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/6547383,city,Cölln an der Spree,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/2950157
5,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/6547383,city,Cölln an der Spree,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/2950159
6,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/6547383,city,Cölln an der Spree,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/6547539
7,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/2950157,city,Cölln an der Spree,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/6547383
8,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/2950157,city,Cölln an der Spree,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/2950159
9,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/2950157,city,Cölln an der Spree,4005728-8,http://www.wikidata.org/entity/Q64,https://sws.geonames.org/6547539


What is up with https://www.geonames.org/2950159/berlin.html vs https://www.geonames.org/6547539/berlin.html???

In [10]:
import pandas as pd

df = pd.read_json("data/data.json")

print(f"Number of BZK cards: {len(df)}")

sampled = df.sample(n=10)
sampled

Number of BZK cards: 1335


,CompensationOffice1,BZKNr,ApplicantFirstName,ApplicantLastName,ApplicantAltFirstName,ApplicantBirthName,ApplicantAltLastName,ApplicantBirthDate,ApplicantBirthPlace,ApplicantCurrentAddress,...,VictimBirthDate,VictimBirthPlace,VictimDeathDate,VictimDeathPlace,VictimCurrentAddress,Heirs,full_response,path,filename,model_name
346,Regierungsbezirksamt für Wiedergutmachung und ...,146478,Szmuel - Leib,Bojarski,None,None,None,18.2.1870,None,"Tel-Aviv, Bar Giorastr. 17",...,1901,Biatystok - Cieptastr.,1943,Vernichtungslager,None,None,"```python\n{\n ""CompensationOffice1"": ""Regi...",/home/bzk-data/1870/,1870_02_18_3_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
422,None,422 349,Deborah,Cremers,None,None,None,10.1.1970,Kichinow (Bessarabion),"rue Notre-Dame-de-Lorette, Paris (Frankreich)",...,None,None,Nov.1941,None,None,None,"```python\n{\n ""CompensationOffice1"": None,...",/home/bzk-data/1870/,1870_01_10_6_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
883,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...",16958 07,None,None,None,None,None,None,None,None,...,24.5.70,Winnert a/d Elbe,None,None,"Hamburg 36, Kaiser-Wilhelmstr. 73",None,"```python\n{\n ""CompensationOffice1"": ""Hans...",/home/bzk-data/1870/,1870_05_24_2_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
21,Bezirksamt für Wiedergutmachung KOBLENZ,434966,Avrum,Abraham,None,None,None,1870,"Vaslui, Rum.","Neuheil-Brak, Hau. Akiva Str. 77, Israel",...,None,None,None,None,None,None,"```python\n{\n ""CompensationOffice1"": ""Bezi...",/home/bzk-data/1870/,1870_00_00_1_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
755,Entschädigungsamt Berlin,152 136,Betty,Falkmann,None,Eisenstadt,None,24.12.1870,Christburg/Westpr.,MexikoDF. Monterrey 327,...,None,None,None,None,None,None,"```python\n{\n ""CompensationOffice1"": ""Ents...",/home/bzk-data/1870/,1870_12_24_1_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
752,Bezirksamt für Wiedergutmachung Köln,330280,Leonova,Gracia,None,None,None,6.5.1870,Nikopol,"Ramatz-Gan, Mosnad Em Frugetr. 59",...,None,None,None,None,None,None,"```python\n{\n ""CompensationOffice1"": ""Bezi...",/home/bzk-data/1870/,1870_05_06_2_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
737,Landesamt für die Wiedergutmachung Stuttgart N...,Es 11759,Sabine,Rosenthal,None,Loewenhaupt,None,5.3.70,Obernbreit/Bay.,Wembley Park Mdx./England,...,5.4.56,Frankfurt-Bockenheim,19.2.51,Middlesex,None,None,"```python\n{\n ""CompensationOffice1"": ""Land...",/home/bzk-data/1870/,1870_03_05_2_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
193,Entschädigungsamt Berlin,12 233,Marie,Kayenberge,None,Kreft,None,24.9.1870,Gransee bei Ruppin,"Berlin-Wittenau, Techowpromenade 56",...,6.2.1876,None,8.3.1953,Berlin-Wittenau,None,None,"```python\n{\n ""CompensationOffice1"": ""Ents...",/home/bzk-data/1870/,1870_09_24_1_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
818,Bayerisches Landesentschädigungsamt,24603,Clara,Hesselberger,None,None,None,28.2.1870,Brünn/Mähren,"3 West 87 Str., New York, U.S.A.",...,None,None,None,None,None,EG AZ: 947381/0728272,"```python\n{\n ""CompensationOffice1"": ""Baye...",/home/bzk-data/1870/,1870_02_28_1_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...
1222,Köln,424 363,Paul,Lehmann,None,None,None,3. April 1870,None,"Bern (Schweiz), Hotel Touring Eigerplatz",...,None,None,None,None,None,None,"```python\n{\n ""CompensationOffice1"": ""Köln...",/home/bzk-data/1870/,1870_04_03_5_0.jpg,InternvlLmdeployModel_InternVL2_5-38B_p-9_fews...


In [33]:
from rapidfuzz import process, fuzz
from rapidfuzz.distance import DamerauLevenshtein
from collections import OrderedDict
import re
import time




PLACE_COLS = [
    "ApplicantBirthPlace", 
    "ApplicantCurrentAddress", 
    "VictimBirthPlace", 
    "VictimDeathPlace",
    "VictimCurrentAddress"
]

PARTS_TO_SKIP = frozenset([ # meaningless and common address parts such as abbreviations for "street"
    "nr",                   # note: Any single letter part will be ignored regardless
    "str",
    "st",
    "street", 
    "strasse", 
    "straße", 
    "apt",
    "av",
    "avenue"
])

MIN_SIMILARITY = 0.85

LVL1_PLACE_SPLIT_REGEX = re.compile(r"[,;/\\]+")
LVL2_PLACE_SPLIT_REGEX = re.compile(r"[\- ]")


POST_PROCESSED_COLS = [f"post_{c}" for c in PLACE_COLS]

def expand_match(row, match, address_part):
    geoname = row["geonameId"]
    r_name = row["name"]
    score = match[1]
    place_type = row["placeType"]
    is_preferred = row["nameType"] == PREFERRED_NAME_URI
    return [geoname, row["stateGeoname"], row["countryGeoname"], place_type, is_preferred, r_name, address_part, score]

MATCH_COL_ORDER = ['geoname', 'stateGeoname', 'countryGeoname', 'isPreferred_city',
                'matchedName_city', 'addressPart_city', 'score_city', 'matched_city',
                'isPreferred_state', 'matchedName_state', 'addressPart_state',
                'score_state', 'matched_state', 'isPreferred_country',
                'matchedName_country', 'addressPart_country', 'score_country',
                'matched_country', 'combinedScore', 'numPreferredLevels',
                'numMatchedLevels', 'combinedMatchedLength', 'numOtherMatches']

def resolve_address(address, place_names_lookup):
    potential_matches = []
    for lvl1_part in [address] + list(reversed(LVL1_PLACE_SPLIT_REGEX.split(address))):
        for part in [lvl1_part] + LVL2_PLACE_SPLIT_REGEX.split(lvl1_part):
            part = part.lower().strip().replace(".", "")
            if len(part) <= 1 or part in PARTS_TO_SKIP or part.isdigit():
                continue # skip common street address parts
            matches = process.extract(part, place_names_lookup["searchKey"], scorer=DamerauLevenshtein.normalized_similarity, score_cutoff=MIN_SIMILARITY)
            for match in matches:
                place_row = place_names_lookup.iloc[match[2]]
                potential_matches.append(expand_match(place_row, match, part))
    if len(potential_matches) == 0:
        return pd.DataFrame()
    col_names = ["geoname", "stateGeoname", "countryGeoname", "placeType", "isPreferred", "matchedName", "addressPart", "score"]
    potential_matches = pd.DataFrame(data=potential_matches, columns=col_names)
    matches_per_type = {}
    for place_type in ["city", "state", "country"]:
        matches = potential_matches[potential_matches["placeType"] == place_type][col_names]
        matches = matches.rename(columns={c: f"{c}_{place_type}" for c in ["isPreferred", "addressPart", "matchedName", "score"]})
        matches[f"matched_{place_type}"] = True
        matches_per_type[place_type] = matches
    combined_matches = pd.concat([
        matches_per_type["city"]
            .merge(matches_per_type["state"].drop(columns=["geoname", "countryGeoname", "placeType"]),
                    on="stateGeoname", how="left")
            .merge(matches_per_type["country"].drop(columns=["geoname", "stateGeoname", "placeType"]),
                    on="countryGeoname", how="left"),
        matches_per_type["state"]
            .merge(matches_per_type["country"].drop(columns=["geoname", "stateGeoname", "placeType"]),
                    on="countryGeoname", how="left"),
        matches_per_type["country"]
    ], verify_integrity=False, ignore_index=True)
    combined_matches["combinedScore"] = combined_matches[["score_city", "score_state", "score_country"]].fillna(1).prod(axis=1)
    combined_matches["numPreferredLevels"] = combined_matches[["isPreferred_city", "isPreferred_state", "isPreferred_country"]].sum(axis=1)
    combined_matches["numMatchedLevels"] = combined_matches[["matched_city", "matched_state", "matched_country"]].sum(axis=1)
    combined_matches["combinedMatchedLength"] = combined_matches[["addressPart_city","addressPart_state","addressPart_country"]].apply(lambda x: sum(len(s) for s in x if isinstance(s, str)), axis=1)
    for sort_col in ["combinedScore", "numPreferredLevels", "numMatchedLevels", "combinedMatchedLength"]:
        max_val = combined_matches[sort_col].max()
        combined_matches = combined_matches[combined_matches[sort_col] == max_val]
    return combined_matches

def resolve_places(row, place_names_lookup):
    post_processed = OrderedDict()
    for colname, cell in row.items():
        combined_matches = pd.DataFrame()
        if cell is not None:
            combined_matches = resolve_address(cell, place_names_lookup)
        if combined_matches.shape[0] > 1:
            selected_row = combined_matches.iloc[0]
            selected_row["numOtherMatches"] = combined_matches.shape[0] - 1
            post_processed[colname] = selected_row.reindex(MATCH_COL_ORDER)
        else:
            post_processed[colname] = pd.Series(None, index = MATCH_COL_ORDER)
    assert len(post_processed) == len(PLACE_COLS), f"Expected {len(PLACE_COLS)} columns, got {len(post_processed)}"
    post_processed = pd.concat(post_processed, ignore_index=True, verify_integrity=False)
    assert isinstance(post_processed, pd.Series)
    return post_processed

print("Loading country and state names...")
print("Processing...")
start = time.monotonic()
post_processed = df[PLACE_COLS].apply(resolve_places, args=(place_names_lookup,), axis=1, result_type="expand")

print(f"Completed for {len(post_processed)} rows in {time.monotonic()-start:.2f}s")

Loading country and state names...
Processing...
Completed for 1335 rows in 47.46s


21 seconds to process 1335 rows should amount to about 9h for all the BZK data.

This is likely increase bz 2 to 4 times once I start considering cities as well.

In [40]:
print(post_processed.shape)
multi_cols = []
for place_col in PLACE_COLS:
    for col in MATCH_COL_ORDER:
        m = col.split("_")
        if len(m) > 1:
            multi_cols.append((place_col, m[1], m[0]))
        else:
            multi_cols.append((place_col, "", col))
post_processed.columns = pd.MultiIndex.from_tuples(multi_cols, names=["addressField", "placeType", "field"])
print(post_processed.columns)

address_merger = (df[PLACE_COLS]
            .stack()
            .reset_index()
)
address_merger = address_merger[address_merger[0].notna() & (address_merger[0].str.strip() != "")]
address_merger.columns = pd.MultiIndex.from_tuples([("rowId",), ("addressField",), ("", "originalAddress")])
print(address_merger.columns)

post_processed1 = post_processed.stack(level=0).rename(columns={"level_0" : "rowId"})
print(post_processed1.columns)
post_processed1 = post_processed1.merge(address_merger, how="inner",
            on=["rowId", "addressField"]
)

post_processed1[[
    "rowId", 
    "geoname", 
    "matchedName", 
    "placeType", 
    ("city", "score"), 
    ("state", "score"), 
    ("country", "score"),
    "numOtherMatches"
]].sample(n=10)

(1335, 115)
MultiIndex([( 'ApplicantBirthPlace',        '',               'geoname'),
            ( 'ApplicantBirthPlace',        '',          'stateGeoname'),
            ( 'ApplicantBirthPlace',        '',        'countryGeoname'),
            ( 'ApplicantBirthPlace',    'city',           'isPreferred'),
            ( 'ApplicantBirthPlace',    'city',           'matchedName'),
            ( 'ApplicantBirthPlace',    'city',           'addressPart'),
            ( 'ApplicantBirthPlace',    'city',                 'score'),
            ( 'ApplicantBirthPlace',    'city',               'matched'),
            ( 'ApplicantBirthPlace',   'state',           'isPreferred'),
            ( 'ApplicantBirthPlace',   'state',           'matchedName'),
            ...
            ('VictimCurrentAddress', 'country',           'isPreferred'),
            ('VictimCurrentAddress', 'country',           'matchedName'),
            ('VictimCurrentAddress', 'country',           'addressPart'),
          

/tmp/ipykernel_79093/3199153664.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  post_processed1 = post_processed.stack(level=0).rename(columns={"level_0" : "rowId"})


ValueError: The column label 'rowId' is not unique.
For a multi-index, the label must be a tuple with elements corresponding to each level.

In [278]:
post_processed.columns = pd.MultiIndex.from_product([
    PLACE_COLS,
    ["Country", "State"],
    ["gndID", "wikidataID", "preferredName", "matchedName", "matchedAddressPart", "similarity"]
])

all_place_fields = post_processed.stack([0, 1], future_stack=True)
all_place_fields = all_place_fields[["wikidataID", "preferredName", "matchedName", "matchedAddressPart", "similarity"]]
all_place_fields = all_place_fields.reset_index().rename(columns = {
        "level_0" : "rowId",
        "level_1" : "placeField",
        "level_2" : "matchedPlaceType"
})

all_place_fields = (
    all_place_fields
    .merge(
        df[PLACE_COLS]
            .stack()
            .reset_index()
            .rename(columns={"level_0" : "rowId", "level_1" : "placeField", 0:"originalAddress"}),
        on = ["rowId", "placeField"],
        how="inner"
    )
)
all_place_fields = all_place_fields[all_place_fields["originalAddress"] != ""]


all_place_matches = all_place_fields[~ all_place_fields["matchedName"].isna()]
unmatched_fields = all_place_fields[all_place_fields["matchedName"].isna()]
unmatched_fields = (unmatched_fields
                    .merge(all_place_matches[["rowId", "placeField"]], on=["rowId", "placeField"], how="left", indicator=True)
                    .query("_merge == 'left_only'").drop(columns="_merge")
)

field_count = len(all_place_fields[["rowId", "placeField"]].drop_duplicates())
match_count = len(all_place_matches[["rowId", "placeField"]].drop_duplicates())

print(f"{match_count} out of {field_count} ({match_count/field_count:.2%}) place fields have been resolved.")
all_place_fields.sample(n=10)

911 out of 2920 (31.20%) place fields have been resolved.


,rowId,placeField,matchedPlaceType,wikidataID,preferredName,matchedName,matchedAddressPart,similarity,originalAddress
4277,904,ApplicantCurrentAddress,State,None,None,None,None,NaN,Au bei Bad Aibling
1344,278,ApplicantBirthPlace,Country,None,None,None,None,NaN,Sulzbach/Saarbrücken
2487,523,ApplicantCurrentAddress,State,None,None,None,None,NaN,Kiriath - Jam - Israel - Giora Joseftal Gimmel...
2845,599,VictimCurrentAddress,State,None,None,None,None,NaN,316 West 94th Street New York 25 NY USA
3198,672,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q414,Argentinien,Argentinien,argentinien,1.0,"Vicente Lopez (B.A.), Argentinien, Calle Mader..."
1489,306,ApplicantBirthPlace,State,None,None,None,None,NaN,Dresden
1623,333,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q64,Berlin,Berlin,berlin,1.0,"Berlin SW 29, Willibald Alexisstr. 28"
4505,951,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q64,Berlin,Berlin,berlin,1.0,"Berlin-Zehlendorf, Auerbachstr. 15"
1917,390,ApplicantCurrentAddress,State,None,None,None,None,NaN,"Giv'at Ascheloscha Malben, Moschav (Beit) Zkenim"
3167,665,ApplicantCurrentAddress,State,None,None,None,None,NaN,"Sao Paulo, Brasilien, Rua Baltazar da Veiga 356"


In [277]:
all_place_matches.merge(all_place_matches[["rowId","placeField"]].drop_duplicates().sample(n=10), how="right")

,rowId,placeField,matchedPlaceType,wikidataID,preferredName,matchedName,matchedAddressPart,similarity,originalAddress
0,808,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q31,Belgien,Belgien,belgien,1.000000,"Brüssel, Belgien - Rue Gaucheret 37"
1,808,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q239,Brüssel,Brussel,brüssel,0.857143,"Brüssel, Belgien - Rue Gaucheret 37"
2,1137,VictimDeathPlace,State,http://www.wikidata.org/entity/Q64,Berlin,Berlin,berlin,1.000000,Berlin (Jüdisches Krankenhaus)
3,359,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q30,USA,USA,usa,1.000000,"Washington 8 D.C./USA, 3420, 29th Street"
4,359,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q61,Washington DC,Washington,washington,1.000000,"Washington 8 D.C./USA, 3420, 29th Street"
5,1162,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q34,Schweden,Schweden,schweden,1.000000,"Radmans Getan 7, Malmö, Schweden"
6,1053,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q30,USA,USA,usa,1.000000,Vineland N.J./USA
7,292,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q31,Belgien,Belgien,belgien,1.000000,Brüssel Belgien rue Gautheret 37
8,292,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q239,Brüssel,Brussel,brüssel,0.857143,Brüssel Belgien rue Gautheret 37
9,635,ApplicantBirthPlace,State,http://www.wikidata.org/entity/Q1055,Hamburg,Hamburg,hamburg,1.000000,Hamburg


In [276]:
partial_matches = all_place_matches[all_place_matches["similarity"] < 1.0]
print(len(partial_matches))
(all_place_matches.merge(partial_matches[["rowId","placeField"]].drop_duplicates().sample(n=10), how="right") if len(partial_matches) > 10 else all_place_matches.merge(partial_matches[["rowId","placeField"]], how="right"))

50


,rowId,placeField,matchedPlaceType,wikidataID,preferredName,matchedName,matchedAddressPart,similarity,originalAddress
0,64,ApplicantBirthPlace,Country,http://www.wikidata.org/entity/Q35,Dänemark,Danemark,dänemark,0.875000,Kopenhagen/Dänemark
1,271,VictimCurrentAddress,Country,http://www.wikidata.org/entity/Q230,Georgien,Georgien,georgen,0.875000,Diesssen a.A. St. Georgen Kirch- / Kreis steig...
2,1333,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q35,Dänemark,Danemark,dänemark,0.875000,Kopenhagen NV./Dänemark Frederikssundsvej 68 G
3,434,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q239,Brüssel,Brussel,brüssel,0.857143,Brüssel (Belgium) 37 rue Gaucheret
4,41,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q96,Mexiko,México,mexico,0.857143,"Mexico, D.F. Adolfo Prieto 914, Colonia del Va..."
5,930,ApplicantBirthPlace,State,http://www.wikidata.org/entity/Q1055,Hamburg,Hamburg,homburg,0.857143,Homburg
6,853,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q155,Brasilien,Brasilien,brasilen,0.888889,San Paulo/Brasilen Rua Baltazar de reiga 356
7,901,ApplicantCurrentAddress,Country,http://www.wikidata.org/entity/Q31,Belgien,Belgien,belgien,1.000000,"Brüssel, Belgien 37 Rue Gautheret"
8,901,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q239,Brüssel,Brussel,brüssel,0.857143,"Brüssel, Belgien 37 Rue Gautheret"
9,1204,ApplicantCurrentAddress,State,http://www.wikidata.org/entity/Q1055,Hamburg,Hamburg,harburg,0.857143,"Holm-Seppenwesen, Krs. Harburg"


In [280]:
unmatched_fields[["rowId", "placeField", "originalAddress"]].sample(n=10)

,rowId,placeField,originalAddress
672,187,ApplicantCurrentAddress,"Johannesburg 46, 8th Avenue, Highlands North"
2542,687,ApplicantCurrentAddress,"Karlsruhe/Baden, Sofienstr. 140"
2834,772,ApplicantCurrentAddress,"Weingartenstr. 2, Celle"
3622,984,ApplicantBirthPlace,"Moren, Krs. Heiligenbeil"
2527,683,ApplicantBirthPlace,Walttram Livland
4515,1235,ApplicantBirthPlace,Skotschau /Österreich.Schlesien
2180,589,ApplicantBirthPlace,Hamm a.d. Sieg/Krs. Waldbröl
1429,378,VictimCurrentAddress,Achuzza Beth Horim Mapustr.
1075,289,ApplicantCurrentAddress,"Imeobackerstrasse 24, T. Storkhoven, Schwerin"
2514,679,VictimBirthPlace,Warschau


# Identified problems

## gnd authority file lacks information about the places or relations between places
- Country and State are classes but there's no class for lower hierachical levels (city, etc.)
    - This is problematic because at that point we cannot distinguish cities from mere streets
- "partOf" relations are very incomplete
    - As an example, there seems to be no property that consistently relates states to the respective country, not even for germany

Possible solution: completement using wikidata

- most gnd places have an owl:sameAs property pointing the respective wikidata id

## Street names may refer to other places
- https://en.wikipedia.org/wiki/Jamaica_Avenue
- Apt. 52, 46 Fort **Washington** Ave., New York City (row 26 applicant current address year 1870)
- New York 33, N.Y./USA 640 Fort **Washington** Ave. (row 5381 applicant current address year 1870)
- Jerusalem, Abessinienstr. 26 b. Klostock matches Abessinien

Considered solutions:

- Use address word order to define disambiguation hierachy
    - Problem: Some adddresses written left to right, others right to left
    - Maybe sort by distance to either end of the address, although this too may be imperfect
- Attempt to locate terms that indicate it's a street
    Problems:
    - Many different languages, terms and abreviations, still risk of missing some cases
    - Address format may not always be that standardized (there may be addresses without such terms)
- Use a more complete KG to create a disambiguation dictionary
    - Not sure if there is such a KG already:
        -  Washington Avenue does not seem to have an entry on wikidata
        -  Jamaica Avenue's entry does not point to Jamaica
- Solve these conflicts manually
    - Problem: Risk of missing some cases
- Use some NLP model to segment the address
    - Problem: time consuming
    - Hamburg, Haifa, Dalia, Israel
 
## Other

- "**Deutschland** deportiert u. verschollen" expanded as deustchland (death place row 234 year 1870)

- ~~Frankfurt/Main and Offenbach/Main resolved to Maine (US State)~~
    - ~~Potential solution: Prioritize matches to german places~~
    - Solved by increasing threshold for similarity
    

- Popular cities (such as capitals) are often written without country information
    - Potential solution (currently in progress): Complete the search list with large cities (based on population? area?) using wikidata (gnd has no city class)

- Countries are sometimes abbreviated (Isr.)

- If a word looks like a state, it can match even if not coherent with the matched country
    - row 492 of year 1870, applicant's current address "Israel, Kfar Malal" matches with ethiopian state "Afar"
    - Potential solution: resolving conflicting information on different levels of location using the best match
 
- There are matches with nations that no longer existed
    - Naharina
    - Probably easy fix, gnd authority contains dates of establishment and termination

# Other noticeable quirks

- dänemark != dänemark, similarity of 0.857
```
> for a, b in zip("dänemark", "dänemark"): print(f"{a} == {b} is {a==b}")
d == d is True
a == ä is False
̈ == n is False
n == e is False
e == m is False
m == a is False
a == r is False
r == k is False
```